In [0]:
import os, sys, time

REPO_ROOT = os.path.dirname(os.getcwd())      # notebooks/ -> repo root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.transforms.common import clean_strings, cast_columns, guard_unique
from src.transforms.gtfs_time import (gtfs_time_to_seconds, day_offset,
                                      seconds_to_clock, service_time_to_ts)

print("imported from", REPO_ROOT)

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
assert spark.conf.get("spark.sql.session.timeZone") == "UTC"
TZ = "America/New_York"          # MBTA's agency_timezone

FEED_VERSION = spark.table("transit.silver.trips").select("_feed_version").first()[0]
st_raw = (spark.table("transit.bronze.gtfs_stop_times")
            .filter(F.col("_feed_version") == FEED_VERSION))

n_raw  = st_raw.count()
n_late = st_raw.filter(gtfs_time_to_seconds("departure_time") >= 86400).count()
print(f"feed_version {FEED_VERSION}: {n_raw:,} stop times, {n_late:,} after midnight "
      f"({100 * n_late / n_raw:.1f}%)")

In [0]:
naive = st_raw.select(
    "trip_id", "stop_sequence", "departure_time",
    F.expr("try_to_timestamp(departure_time, 'HH:mm:ss')").alias("naive_ts"))

(naive.select(
    F.count("*").alias("rows"),
    F.sum(F.col("departure_time").isNull().cast("int")).alias("blank_at_source"),
    F.sum(F.col("naive_ts").isNull().cast("int")).alias("naive_nulls"))
 .display())

(naive.filter(F.col("naive_ts").isNull() & F.col("departure_time").isNotNull())
   .orderBy(F.desc("departure_time")).limit(10).display())

try:
    st_raw.agg(F.max(F.to_timestamp("departure_time", "HH:mm:ss"))).collect()
    print("plain to_timestamp: no error -> late-night trips were silently nulled")
except Exception as e:
    print("plain to_timestamp FAILED THE WHOLE QUERY:", type(e).__name__)
    print(str(e)[:250])

In [0]:
TYPES = {"stop_sequence": "INT", "pickup_type": "INT", "drop_off_type": "INT",
         "timepoint": "INT", "shape_dist_traveled": "DOUBLE",
         "continuous_pickup": "INT", "continuous_drop_off": "INT"}

st, fails, missing = cast_columns(clean_strings(st_raw), TYPES)
print("cast failures:", fails)
print("spec columns not in data:", missing or "none")
assert sum(fails.values()) == 0, f"cast failures: {fails}"

st = (st
  .withColumn("arrival_seconds",      gtfs_time_to_seconds("arrival_time"))
  .withColumn("departure_seconds",    gtfs_time_to_seconds("departure_time"))
  .withColumn("arrival_day_offset",   day_offset("arrival_seconds"))
  .withColumn("departure_day_offset", day_offset("departure_seconds"))
  .withColumn("departure_clock",      seconds_to_clock("departure_seconds")))

p = st.select(
    F.sum((F.col("arrival_time").isNotNull()   & F.col("arrival_seconds").isNull()).cast("int")).alias("arrival_parse_failures"),
    F.sum((F.col("departure_time").isNotNull() & F.col("departure_seconds").isNull()).cast("int")).alias("departure_parse_failures"),
    F.sum(F.col("arrival_time").isNull().cast("int")).alias("arrival_blank_at_source"),
    F.sum(F.col("departure_time").isNull().cast("int")).alias("departure_blank_at_source"),
    F.max("departure_time").alias("latest_time")).collect()[0].asDict()
print(p)
assert p["arrival_parse_failures"] == 0 and p["departure_parse_failures"] == 0, "unparseable times"

st, removed = guard_unique(st, ["trip_id", "stop_sequence"])
print("duplicate (trip_id, stop_sequence) removed:", removed)

In [0]:
w = Window.partitionBy("trip_id").orderBy("stop_sequence")

(st.withColumn("prev_departure", F.lag("departure_seconds").over(w))
   .select(
       F.sum((F.col("arrival_seconds") > F.col("departure_seconds")).cast("int")).alias("arrives_after_it_departs"),
       F.sum((F.col("departure_seconds") < F.col("prev_departure")).cast("int")).alias("goes_backwards_in_time"),
       F.max("departure_day_offset").alias("max_day_offset"))
   .display())

In [0]:
t0 = time.time()
(st.withColumn("_silver_loaded_at", F.current_timestamp())
   .write.format("delta").mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("transit.silver.stop_times"))
write_secs = time.time() - t0

d = spark.sql("DESCRIBE DETAIL transit.silver.stop_times").select("numFiles", "sizeInBytes").first()
print(f"silver.stop_times: {spark.table('transit.silver.stop_times').count():,} rows · "
      f"{write_secs:.0f}s write · {d.numFiles} files · {d.sizeInBytes/1e6:.0f} MB")

In [0]:
dates = (spark.createDataFrame([("2026-09-22",), ("2026-11-01",)], "d string")
           .select(F.to_date("d").alias("service_date")))

# How far is "noon minus 12h" from midnight on each date?
(dates.select("service_date",
    ((F.unix_timestamp(F.to_utc_timestamp(
        F.concat(F.date_format("service_date", "yyyy-MM-dd"), F.lit(" 12:00:00")).cast("timestamp"), TZ))
      - 43200)
     - F.unix_timestamp(F.to_utc_timestamp(F.col("service_date").cast("timestamp"), TZ))
    ).alias("anchor_minus_midnight_seconds"))
 .display())

# One real late-night trip, resolved both ways on both dates
late = (spark.table("transit.silver.stop_times")
          .filter("departure_day_offset >= 1")
          .orderBy(F.desc("departure_seconds"))
          .select("trip_id", "stop_id", "departure_time", "departure_seconds").limit(1))

(late.crossJoin(dates)
   .withColumn("correct_local", F.from_utc_timestamp(
        service_time_to_ts("service_date", "departure_seconds", TZ), TZ))
   .withColumn("midnight_based_local", F.from_utc_timestamp(
        F.timestamp_seconds(F.unix_timestamp(F.to_utc_timestamp(
            F.col("service_date").cast("timestamp"), TZ)) + F.col("departure_seconds")), TZ))
   .display())

In [0]:
s = spark.table("transit.silver.stop_times")

n_silver   = s.count()
n_late_out = s.filter("departure_seconds >= 86400").count()
unexplained = s.filter(F.col("departure_time").isNotNull() & F.col("departure_seconds").isNull()).count()
offset_ok  = s.filter("departure_day_offset >= 1").count() == n_late_out

print(f"bronze {n_raw:,} · silver {n_silver:,}")
print(f"after-midnight stop times: {n_late_out:,} (bronze said {n_late:,})")
print(f"unexplained null departures: {unexplained}")

assert n_silver == n_raw,      "rows lost between bronze and silver"
assert unexplained == 0,       "parser produced nulls"
assert n_late_out == n_late,   "late-night count changed between bronze and silver"
assert offset_ok,              "day_offset disagrees with seconds"
print("\n2.2 done")